## Import libraries

In [1]:
import tomopy
from helperFunctions import MoviePlotter
from tomoDataClass import tomoData
from alignment_methods import reprojection_consistency_score
import h5py
import numpy as np
from skimage.transform import pyramid_gaussian
from scipy.ndimage import zoom

def tomo_data(file,redo_align=False):
    try:
        with h5py.File(file) as hf:
            projs = hf['data'][...]
            angles = hf['angles'][...]
    except KeyError:
        with h5py.File(file) as hf:
            projs = hf['data'][...]
            angles = hf['angles'][...]
    angles = angles * np.pi / 180
    #Shift angles to be centered around 0
    angles = angles - np.mean(angles)
    return projs, angles


PyTorch imported successfully.
SVMBIR imported successfully.


## Import real Data

In [2]:
import os

downsample = 4
recon_alg = 'gridrec'
# filename = "/Users/levihancock/Library/CloudStorage/Box-Box/BYU_CXI_Research_Team/ProjectFolders/IFE-STAR/IFE-Ptycho-Tomo/APS_2ID_GUP1013052_August_2025/levi_tomoReconstructions/tomo_data_run_final_2.hdf5"
filename = "/home/ljh79/groups/grp_ptychi/nobackup/autodelete/Oct2025APSdata/tomo_data_run_final_2.hdf5"

cached_filename = filename.replace(".hdf5", f"_ds{downsample}.hdf5")

if os.path.exists(cached_filename):
    print(f"Loading downsampled cache: {cached_filename}")
    projections, angles = tomo_data(cached_filename)
    print(projections.shape)
    tomo = tomoData(projections, angles)
else:
# if True:
    print("Cache not found — loading full dataset...")
    projections_og, angles = tomo_data(filename, redo_align=True)
    print("Full dataset shape:", projections_og.shape)

    #Remove projection and angle 26
    projections_og = np.delete(projections_og, 26, axis=0)
    angles = np.delete(angles, 26)

    #Remove projection and angle 19
    projections_og = np.delete(projections_og, 19, axis=0)
    angles = np.delete(angles, 19)
    _needs_downsample = True

if not os.path.exists(cached_filename):
# if True:
    print("Downsampling...")
    projections = zoom(projections_og, (1, 1/downsample, 1/downsample), order=1)
    print(projections.shape)

    print(f"Saving downsampled cache to: {cached_filename}")
    with h5py.File(cached_filename, 'w') as hf:
        hf.create_dataset('data', data=projections)
        hf.create_dataset('angles', data=angles * 180 / np.pi)  # save back in degrees

    tomo = tomoData(projections, angles)

num_angles = projections.shape[0] if os.path.exists(cached_filename) else projections_og.shape[0]
print(f"Number of angles: {num_angles}")

Loading downsampled cache: /home/ljh79/groups/grp_ptychi/nobackup/autodelete/Oct2025APSdata/tomo_data_run_final_2_ds4.hdf5
(556, 146, 452)
Number of angles: 556


In [ ]:
print(projections.shape)
tomo.makeNotebookProjMovie()

(556, 146, 452)


Output()

: 

## Align Data

In [ ]:
# # Show bad reconstruction prior to alignment
# tomo.reset_workingProjections(x_size=None, y_size=None) #You can adjust these for tighter cropping
# tomo.normalize(isPhaseData=True)

# tomo.reconstruct(algorithm=recon_alg)
# print("\nBad reconstruction prior to alignment")
# badRecon = tomo.recon.copy()
# MoviePlotter(badRecon)

In [ ]:
# # Reprojection Consistency Score — before alignment
# rcs_before, _, _ = tomo.reprojection_consistency_score(plot=False)

# #Sinogram consistency score — before alignment
# scs_before, _, _, _, _ = tomo.sinogram_consistency_score(plot=False)

In [ ]:
tomo.reset_workingProjections(x_size=None, y_size=None) #You can adjust these for tighter cropping
tomo.normalize(isPhaseData=True)
# PMA - 3 levels, scale=2, of_sigma=2.0, stepRatio=0.8

#set xROI range to be width of the total number of angles divided by downsample
maxWidth = tomo.num_angles // downsample
xROI_Range = [tomo.workingProjections.shape[2] // 2 - maxWidth, tomo.workingProjections.shape[2] // 2 + maxWidth]
tomo.PMA(
    levels=3, scale=2, iterations_per_level=[5, 5, 5],
    tolerance=0.01, algorithm=recon_alg, standardize=False,
    shift_method='optical_flow', of_sigma=2.0, stepRatio=0.8, plot=True, xROI_Range=xROI_Range, yROI_Range=[0, tomo.workingProjections.shape[1] - (300//downsample)]
)

tomo.make_updates_shift()
tomo.makeNotebookProjMovie(show_trust_region=True)
tomo.displayWorkingSinogram(row_index=tomo.workingProjections.shape[0] // downsample // 2)




Normalizing projections


Projection Matching Alignment (PMA) [optical_flow | matching_preprocess]
Centering Projections
Original center: 225.5
Center of frame: 226
Aligned projections shifted by 0.5 pixels
Projections are currently centered at pixel 225.5. Residual offset: 0.5

--- PMA Level 2 (4x downsampled, 5 iterations) ---
Using ROI: x=[87, 365], y=[0, 71] (downsampled by 4x)
H and W values are 36 and 113
Downsample ROI bounds are x=21 to 91, y=0 to 17


PMA Level 2 iterations:   0%|          | 0/5 [00:00<?, ?it/s]Error.  nthreads cannot be larger than environment variable "NUMEXPR_MAX_THREADS" (64)Error.  nthreads must be a positive integerWarning: Unable to get available GPU memory. Defaulting to 1GB.
Error: allocateGPUMemory malloc3d: CUDA error 46: all CUDA-capable devices are busy or unavailable.
Error: Error allocating GPU memory
Error: allocateGPUMemory malloc3d: CUDA error 46: all CUDA-capable devices are busy or unavailable.
Error: Error allocating GPU memory


In [ ]:
tomo.reset_workingProjections(x_size=None, y_size=None) #You can adjust these for tighter cropping
tomo.normalize(isPhaseData=True)

# XCA Pass 1 - coarse (downsample=4)
tomo.cross_correlate_align(
    tolerance=0.001, maxShiftTolerance=0.5, max_iterations=5, stepRatio=0.8,
    downsample=4, use_grad=True,
    yROI_Range=None, xROI_Range=None, isFull360=False, plot=True
)
tomo.center_projections()
tomo.make_updates_shift()
tomo.makeNotebookProjMovie(show_trust_region=True)
tomo.displayWorkingSinogram(row_index=tomo.workingProjections.shape[0] // downsample // 2)

In [ ]:

# XCA Pass 2 - medium
tomo.cross_correlate_align(
    tolerance=0.001, maxShiftTolerance=0.5, max_iterations=5, stepRatio=0.8,
    downsample=2, use_grad=True,
    yROI_Range=[0, tomo.workingProjections.shape[1] - (200//downsample)], xROI_Range=[250//downsample, tomo.workingProjections.shape[2] - (250 // downsample)], isFull360=False, plot=True
)
tomo.make_updates_shift()
tomo.makeNotebookProjMovie(show_trust_region=True)
tomo.displayWorkingSinogram(row_index=tomo.workingProjections.shape[0] // downsample // 2)

In [ ]:

tomo.cross_correlate_align(
    tolerance=0.001, maxShiftTolerance=0.5, max_iterations=5, stepRatio=0.8,
    downsample=1, use_grad=True,
    yROI_Range=[0, tomo.workingProjections.shape[1] - (300//downsample)], xROI_Range=[250//downsample, tomo.workingProjections.shape[2] - (250 // downsample)], isFull360=False, plot=True
)
tomo.make_updates_shift()
tomo.makeNotebookProjMovie(show_trust_region=True)
tomo.displayWorkingSinogram(row_index=tomo.workingProjections.shape[0] // downsample // 2)

In [ ]:

# PMA - 3 levels, scale=2, of_sigma=2.0, stepRatio=0.8

#set xROI range to be width of the total number of angles divided by downsample
maxWidth = tomo.num_angles//2 // downsample
xROI_Range = [tomo.workingProjections.shape[2] // 2 - maxWidth, tomo.workingProjections.shape[2] // 2 + maxWidth]
tomo.PMA(
    levels=3, scale=2, iterations_per_level=[5, 5, 5],
    tolerance=0.01, algorithm=recon_alg, standardize=False,
    shift_method='optical_flow', of_sigma=2.0, stepRatio=0.8, plot=True, xROI_Range=xROI_Range, yROI_Range=[0, tomo.workingProjections.shape[1] - (300//downsample)]
)

tomo.make_updates_shift()
tomo.makeNotebookProjMovie(show_trust_region=True)
tomo.displayWorkingSinogram(row_index=tomo.workingProjections.shape[0] // downsample // 2)


In [ ]:
tomo.makeNotebookProjMovie()
tomo.reconstruct(algorithm=recon_alg)
print("\nGood Reconstruction after alignment")
tomo.makeNotebookReconMovie()

In [ ]:
tomo.displayReconOrthogonalSlices()

In [ ]:
# Reprojection Consistency Score — after alignment
rcs_after, _, _ = tomo.reprojection_consistency_score(plot=True)

#Sinogram consistency score — after alignment
scs_after, _, _, _, _ = tomo.sinogram_consistency_score(plot=True)

#Fourier shell correlation resolution - after alignment
res_after = tomo.fourier_shell_correlation(algorithm=recon_alg, plot=True)

# # print("=" * 52)
# # print("  ALIGNMENT QUALITY SUMMARY")
# # print("=" * 52)
# # print(f"  {'Metric':<22} {'Before':>8} {'After':>8} {'Δ':>10}")
# # print(f"  {'-'*50}")
# # print(f"  {'RCS (lower = better)':<22} {rcs_before:>8.4f} {rcs_after:>8.4f} {rcs_before - rcs_after:>+10.4f}")
# # print("=" * 52)

In [ ]:
# import tifffile
# tifffile.imwrite("reconstruction(downsampled).tiff", tomo.recon.astype("float32"))